# [미지 구간 평가 · 2026-10-26] β=30 정책을 한 번도 쓰지 않은 2026-07-14 이후 데이터로 평가

기존 데이터(학습·검증·테스트)는 2026-07-13에서 끝난다. 그 이후 구간은 하이브리드 임베딩 모델(2025-06까지 학습),
PPO 풀(2025년 학습), 모든 하이퍼파라미터 선택, 지금까지의 감사에서 **한 번도 쓰이지 않았다.**
이 노트북은 새 구간 데이터를 받아 기존과 똑같은 파이프라인으로 피처·임베딩을 만들고, 배포 중인 β=30을 **한 번만** 평가한다.

## 사전 등록한 평가 기준 (실행 전에 고정 — 결과를 보고 바꾸지 않는다)
- **평가 구간**: 2026-07-14 00:00 UTC ~ 받은 데이터의 마지막 버킷. **평가 방식**: 2주 워밍업(`crypto_start_state_audit_rt1006`와 동일), α=0.01, top-K=10, 비용 0.1%.
- **주 지표**: β=30의 총수익률·샤프·MDD.
- **비교 대상**: (a) BTC를 β=30의 노출 경로대로 보유, (b) 44종목 등가중을 β=30의 노출 경로대로 보유, (c) BTC 보유, (d) 등가중(비용 포함).
- **판정**: β=30이 (a)와 (b) **둘 다**보다 총수익률이 높으면 "같은 노출 대비 종목 선택 우위가 미지 구간에서도 유지됐다", 아니면 "유지되지 않았다"로 보고한다.
- 보조: 4개 β 중 β=30의 순위. **튜닝·재실행으로 결과를 고르지 않는다.**

## 실행
- **런타임 유형을 GPU(T4)로 바꾼 뒤** `런타임 > 모두 실행`. 약 30~60분(시세 다운로드 ~5분, 임베딩 ~10~20분).
- 드라이브 연결 창에서 **ksbdaniel7@gmail.com** 계정을 고른다.
- 결과: `data/crypto/rt1006/crypto_holdout_eval.csv` / 받은 원시 시세: `data/crypto/holdout_raw/` / 새 임베딩: `data/crypto/rt1006/crypto_rl_embeddings_holdout.h5`
- 새 구간 데이터와 기존 데이터가 겹치는 2026-06-25~07-12에서 **피처·임베딩이 기존 것과 일치하는지 먼저 검증**하고, 어긋나면 평가 전에 멈춘다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow chronos-forecasting einops


In [ ]:
# 실행기 — GitHub의 원본 노트북(커밋 0e86a41 고정)을 불러와 셀을 순서대로 실행한다.
# (1) 설정 셀 직후 모델·결과 경로를 새 폴더(rt1006)로 바꿔 기존 결과를 덮어쓰지 않는다.
# (2) 모든 .learn() 호출 결과를 캐시에 저장 — 끊겨도 `모두 실행`을 다시 누르면 학습된 부분은 건너뛴다.
import json, urllib.request
from pathlib import Path
COMMIT = '0e86a41a701841d9a17398a8cebf240cd0c40218'
RAW = f'https://raw.githubusercontent.com/winbean-kwon/ToBeRich/{COMMIT}/'
TAG = 'rt1006'
_ip = get_ipython()

def fetch_cells(path):
    nb = json.loads(urllib.request.urlopen(RAW + path).read().decode('utf-8'))
    return {i: (c['source'] if isinstance(c['source'], str) else ''.join(c['source']))
            for i, c in enumerate(nb['cells']) if c['cell_type'] == 'code'}

PATCH = r"""
MODEL_DIR = DRIVE_ROOT / 'models' / TAG
CRYPTO_DIR = DATA_DIR / 'crypto' / TAG
MODEL_DIR.mkdir(parents=True, exist_ok=True); CRYPTO_DIR.mkdir(parents=True, exist_ok=True)
POOL_DIR = MODEL_DIR / 'crypto_hrl_pool'
ORIG_PPO_PATH = MODEL_DIR / 'crypto_ppo_portfolio.zip'
for _n in ('EIIE_MODEL_DIR', 'LSRE_CAAN_MODEL_DIR'):
    if _n in globals():
        globals()[_n] = MODEL_DIR / Path(globals()[_n]).name
        globals()[_n].mkdir(parents=True, exist_ok=True)
CACHE = MODEL_DIR / '_learn_cache'; CACHE.mkdir(exist_ok=True)
import stable_baselines3 as _sb3
_BASE_PPO, _BASE_DQN = _sb3.PPO, _sb3.DQN
def _cached(base):
    class _C(base):
        def learn(self, total_timesteps, *a, **k):
            _CTR[0] += 1
            p = CACHE / f'{_NB[0]}_c{_CUR[0]}_{_CTR[0]}_{base.__name__}.zip'
            if p.exists():
                self.set_parameters(str(p), exact_match=True, device=self.device)
                self.num_timesteps += total_timesteps
                print(f'[캐시] {p.name} 로드 — 학습 생략')
                return self
            out = super().learn(total_timesteps, *a, **k)
            self.save(str(p))
            return out
    _C.__name__ = _C.__qualname__ = base.__name__
    return _C
PPO, DQN = _cached(_BASE_PPO), _cached(_BASE_DQN)
print(f'[경로] 모델 → {MODEL_DIR} | 결과 CSV → {CRYPTO_DIR}')
"""

_NB, _CUR, _CTR = [''], [0], [0]

def run_cells(path, ids, patch_after, extra=None, subs=None):
    """extra: 경로 변경 직후 실행할 코드(예: β 변경). subs: {셀번호: [(원문, 대체), ...]} 문자열 치환."""
    cells = fetch_cells(path)
    _NB[0] = Path(path).stem
    for i in ids:
        _CUR[0], _CTR[0] = i, 0
        src = cells[i]
        for old, new in (subs or {}).get(i, []):
            assert old in src, f'셀 {i}에 치환 대상 없음: {old!r}'
            src = src.replace(old, new)
        print(f'\n########## {Path(path).name} 셀 {i} ##########')
        r = _ip.run_cell(src)
        if not r.success:
            raise RuntimeError(f'{path} 셀 {i}에서 오류 — 위 오류 메시지를 확인하세요')
        if i == patch_after:
            for code in (PATCH, extra):
                if code and not _ip.run_cell(code).success:
                    raise RuntimeError('경로/설정 변경 실패')
print('실행기 준비 완료')


In [ ]:
# 준비: 기존 데이터(임베딩 h5·features_5m)·환경·평가 함수 — 원본 crypto_beta_robustness_colab(커밋 0e86a41) 셀 재사용
run_cells('crypto/notebooks/crypto_beta_robustness_colab.ipynb', [3, 5, 6, 8, 10, 12, 13, 15, 17], patch_after=3, extra="BETA = 30   # 재학습 후 검증 샤프 최고 β\nPOOL_DIR_ROBUST = MODEL_DIR / 'crypto_beta_robustness'\nDEPLOYED_AGENT_PATH = MODEL_DIR / 'crypto_hrl_pool_v4' / f'agent_beta_{BETA}.zip'\nassert DEPLOYED_AGENT_PATH.exists(), f'재학습한 v4 β={BETA} 모델 없음: {DEPLOYED_AGENT_PATH}'\nprint(f'[β 변경] BETA={BETA} | 기준 모델: {DEPLOYED_AGENT_PATH}')\n")


## 1. 새 구간 데이터 준비

In [ ]:
# 저장소 코드(피처 계산·하이브리드 모델) 불러오기 — 기존 파이프라인과 같은 코드를 쓰기 위해 커밋 고정
import sys, subprocess
REPO = Path('/content/ToBeRich')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/winbean-kwon/ToBeRich.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', '55bce50d4e3f7049177bdf43d4f7c43f4bbe9772'], check=True)
sys.path.insert(0, str(REPO))
from crypto.preprocess_features import FEATURE_COLS as PF_FEATURE_COLS, add_features
from crypto.models.hybrid_model import load_hybrid_model, load_chronos_pipeline, extract_chronos_embs
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| 피처', len(PF_FEATURE_COLS), '개')
if DEVICE == 'cpu':
    print('⚠️ GPU가 아니면 임베딩 계산이 수 시간 걸릴 수 있다 — 런타임 유형을 GPU로 바꾸는 것을 권장')


In [ ]:
%%time
# 새 구간 5분봉 다운로드 (바이낸스 공개 시세 전용 서버 — API 키 불필요, 해외 IP 제한 없음)
import time, json as _json, urllib.request, urllib.error
RAW_DIR = DATA_DIR / 'crypto' / 'holdout_raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)
DL_START = pd.Timestamp('2026-06-01', tz='UTC')
HOSTS = ['https://data-api.binance.vision', 'https://api.binance.com']


def fetch_klines(sym, start_ms, end_ms):
    rows, cur = [], start_ms
    while cur < end_ms:
        q = f'/api/v3/klines?symbol={sym}&interval=5m&startTime={cur}&limit=1000'
        for host in HOSTS:
            try:
                batch = _json.loads(urllib.request.urlopen(host + q, timeout=30).read())
                break
            except urllib.error.HTTPError as e:
                if e.code == 400:   # 존재하지 않는(상장폐지된) 심볼
                    return rows
                last_err = e
            except Exception as e:
                last_err = e
        else:
            raise RuntimeError(f'{sym} 다운로드 실패: {last_err}')
        if not batch:
            break
        rows.extend(batch)
        nxt = batch[-1][0] + 5 * 60 * 1000
        if nxt <= cur:
            break
        cur = nxt
        time.sleep(0.05)
    return rows


now_ms = int(time.time() * 1000)
raw = {}
for sym in symbols:
    path = RAW_DIR / f'{sym}.parquet'
    if path.exists():
        raw[sym] = pd.read_parquet(path)
        continue
    rows = fetch_klines(sym, int(DL_START.timestamp() * 1000), now_ms)
    df = pd.DataFrame([r[:6] for r in rows], columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
    df[['open', 'high', 'low', 'close', 'volume']] = df[['open', 'high', 'low', 'close', 'volume']].astype(float)
    df = df[df['timestamp'] + 5 * 60 * 1000 <= now_ms]   # 진행 중인 마지막 봉 제외
    df.to_parquet(path, index=False)
    raw[sym] = df
    print(f'{sym}: {len(df):,}봉', '' if len(df) else '(데이터 없음 — 상장폐지/거래정지)')
print('다운로드 완료:', sum(len(d) > 0 for d in raw.values()), '/', len(symbols), '심볼')


In [ ]:
# 피처 계산 — preprocess_features.load_and_regularize/add_features와 같은 방식(5분 그리드 정합화 → 지표 → 타겟)
def regularize(df):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
    df = df.drop_duplicates(subset='datetime').set_index('datetime').sort_index()
    df = df.reindex(pd.date_range(df.index[0], df.index[-1], freq='5min', tz='UTC'))
    df['close'] = df['close'].ffill()
    for col in ('open', 'high', 'low'):
        df[col] = df[col].fillna(df['close'])
    df['volume'] = df['volume'].fillna(0.0)
    return df.drop(columns=['timestamp'])


feats = {}
for sym, df in raw.items():
    if len(df) < 200:
        continue
    f = add_features(regularize(df)).dropna(subset=['SMA_60', 'Volatility_20b'])  # 마지막 6봉은 타겟만 NaN으로 둔다
    feats[sym] = f

# 검증 1: 기존 features_5m과 겹치는 구간(2026-06-25 ~ 07-12)에서 값 비교 — EMA/RSI는 시작점이 달라도 수백 봉 뒤엔 같아진다
OV_START, OV_END = pd.Timestamp('2026-06-25', tz='UTC'), pd.Timestamp('2026-07-12', tz='UTC')
worst = []
for sym in list(feats)[:44]:
    old_path = FEAT_DIR / f'{sym}.parquet'
    if not old_path.exists():
        continue
    old = pd.read_parquet(old_path, columns=['datetime'] + PF_FEATURE_COLS + ['RetTarget_6b']).set_index('datetime')
    old = old.loc[OV_START:OV_END]
    new = feats[sym].loc[OV_START:OV_END]
    idx = old.index.intersection(new.index)
    if len(idx) < 100:
        continue
    rel = ((new.loc[idx, PF_FEATURE_COLS] - old.loc[idx, PF_FEATURE_COLS]).abs() / (old.loc[idx, PF_FEATURE_COLS].abs() + 1e-9)).max()
    worst.append((sym, float(rel.max()), rel.idxmax()))
worst.sort(key=lambda x: -x[1])
print('겹치는 구간 피처 최대 상대오차 상위 5:', [(s, f'{v:.2e}', c) for s, v, c in worst[:5]])
FEAT_OK = len(worst) >= 30 and worst[0][1] < 1e-3
print('피처 일치 검증:', 'OK' if FEAT_OK else '⚠️ 불일치 — 아래 임베딩 검증 결과까지 보고 판단')


## 2. 임베딩 (기존 h5와 일치 검증 후 진행)

In [ ]:
%%time
# 30분 버킷 임베딩 — 기존 h5와 같은 '심볼별 위상'(윈도우 끝 봉 시각 mod 1800초)을 그대로 따른다
hybrid = load_hybrid_model(DRIVE_ROOT / 'models' / 'crypto_hybrid_best.pt', device=DEVICE)  # 기존 h5를 만든 그 모델
pipeline = load_chronos_pipeline(device=DEVICE)
SEQ = 60
EMB_FROM = pd.Timestamp('2026-06-25', tz='UTC')   # 겹침 검증(06-25~07-12) + 워밍업(07-01~) + 평가(07-14~)

phase = {}
for s, t in zip(symbols_list, ts_list):
    phase.setdefault(s, t % BUCKET_SEC)

new_sym, new_ts, new_emb = [], [], []
for sym, f in feats.items():
    if sym not in phase:
        continue
    ts_sec = f.index.asi8 // 10**9
    feat = f[PF_FEATURE_COLS].to_numpy(np.float32)
    close = f['close'].to_numpy(np.float32)
    cand = [i for i in range(SEQ - 1, len(f)) if ts_sec[i] % BUCKET_SEC == phase[sym] and f.index[i] >= EMB_FROM]
    for k in range(0, len(cand), 512):
        b = cand[k:k + 512]
        Xb = np.empty((len(b), SEQ, len(PF_FEATURE_COLS)), np.float32)
        Cb = np.empty((len(b), SEQ), np.float32)
        for j, i in enumerate(b):
            w = feat[i - SEQ + 1:i + 1]
            Xb[j] = (w - w.mean(0)) / (w.std(0) + 1e-8)
            Cb[j] = close[i - SEQ + 1:i + 1]
        c_emb = torch.tensor(extract_chronos_embs(pipeline, Cb, batch_size=512, device=DEVICE), device=DEVICE)
        with torch.no_grad():
            e = hybrid.embed(torch.tensor(Xb, device=DEVICE), c_emb).cpu().numpy()
        new_sym += [sym] * len(b); new_ts += [int(ts_sec[i]) for i in b]; new_emb.append(e)
new_emb = np.concatenate(new_emb).astype(np.float32)
print(f'새 임베딩 {len(new_emb):,}개')

# 검증 2: 겹치는 구간에서 기존 h5 임베딩과 코사인 유사도
cos = []
for s, t, e in zip(new_sym, new_ts, new_emb):
    idx = emb_index.get((s, t // BUCKET_SEC))
    if idx is not None:
        o = embs_all[idx]
        cos.append(float(np.dot(o, e) / (np.linalg.norm(o) * np.linalg.norm(e) + 1e-12)))
cos = np.array(cos)
print(f'겹치는 임베딩 {len(cos):,}개 | 코사인 유사도 평균 {cos.mean():.5f} · 최소 {cos.min():.5f}')
EMB_OK = len(cos) > 1000 and cos.mean() > 0.999
assert EMB_OK, '새로 만든 임베딩이 기존 h5와 일치하지 않는다 — 파이프라인이 다르므로 평가하지 않고 멈춘다'
print('임베딩 일치 검증: OK')

with h5py.File(CRYPTO_DIR / 'crypto_rl_embeddings_holdout.h5', 'w') as hf:
    hf.create_dataset('symbols', data=np.array(new_sym, dtype='S12'), compression='gzip')
    hf.create_dataset('timestamps', data=np.array(new_ts, dtype=np.int64), compression='gzip')
    hf.create_dataset('embeddings', data=new_emb, compression='gzip')


In [ ]:
# 기존 인덱스에 새 구간만 추가(기존 키는 덮어쓰지 않는다) → 평가 버킷 확정
base = len(embs_all)
add_e, n_new = [], 0
for s, t, e in zip(new_sym, new_ts, new_emb):
    key = (s, t // BUCKET_SEC)
    if key in emb_index:
        continue
    emb_index[key] = base + len(add_e); add_e.append(e)
embs_all = np.concatenate([embs_all, np.asarray(add_e, np.float32)]) if add_e else embs_all

new_ts_set = {}
for s, t in zip(new_sym, new_ts):
    new_ts_set.setdefault(s, set()).add(t)
n_ret = 0
for sym, f in feats.items():
    ok = new_ts_set.get(sym, set())
    for t, r in zip(f.index.asi8 // 10**9, f['RetTarget_6b'].to_numpy()):
        t = int(t)
        key = (sym, t // BUCKET_SEC)
        if t in ok and not np.isnan(r) and key not in ret_index:
            ret_index[key] = float(r); n_ret += 1
env_kwargs.update(emb_index=emb_index, embs_all=embs_all, ret_index=ret_index)

all_buckets = sorted(set(all_buckets) | {t // BUCKET_SEC for t in new_ts})
HOLDOUT_START = pd.Timestamp('2026-07-14', tz='UTC')
ret_buckets = {b for (_, b) in ret_index}
holdout_buckets = [b for b in all_buckets if b * BUCKET_SEC >= HOLDOUT_START.timestamp() and b in ret_buckets]
print(f'추가: 임베딩 {len(add_e):,}개 · 수익률 {n_ret:,}개')
print(f'평가 구간: {bucket_to_ts(holdout_buckets[0])} ~ {bucket_to_ts(holdout_buckets[-1])} ({len(holdout_buckets):,}스텝 ≈ {len(holdout_buckets)/48:.0f}일)')
assert holdout_buckets[0] * BUCKET_SEC - HOLDOUT_START.timestamp() < 3600, '평가 시작 버킷이 비어 있다'


## 3. 사전 등록한 평가 (한 번만)

In [ ]:
%%time
# 사전 등록한 평가 — 한 번만 실행한다
WARMUP = 14 * 48
BETAS = [-90, -30, 30, 90]
POOL_V4 = MODEL_DIR / 'crypto_hrl_pool_v4'
models = {b: PPO.load(str(POOL_V4 / f'agent_beta_{b}.zip')) for b in BETAS}


def run_policy(model, buckets, start='warmup', alpha=ALPHA, top_k=TOP_K):
    """crypto_start_state_audit_rt1006의 run_policy와 동일."""
    if start == 'warmup':
        i0 = all_buckets.index(buckets[0])
        lo = max(0, i0 - WARMUP)
        run_bk, skip = all_buckets[lo:i0] + list(buckets), i0 - lo
    else:
        run_bk, skip = list(buckets), 0
    env = SparseGatedCryptoPortfolioEnv(buckets=run_bk, top_k=top_k, **env_kwargs)
    obs, _ = env.reset()
    if start in ('cash', 'warmup'):
        env.weights = np.zeros(env.n_assets, dtype=np.float32)
        obs = env._get_obs()
    expo, done = [], False
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        tw, rho, top_idx = env.action_to_weights(a)
        w = (1 - alpha) * env.weights + alpha * tw
        expo.append(float(w.sum()))
        obs, _, done, _, info = env.step_with_weights(w, extra_info={'rho': rho, 'selected': top_idx.tolist()})
    vals = np.asarray(env.value_history[skip:], dtype=np.float64)
    return vals / vals[0], list(env.turnover_history[skip:]), np.asarray(expo[skip:])


bk = holdout_buckets
r_ew = np.asarray([np.mean([ret_index[(s, b)] for s in symbols if (s, b) in ret_index] or [0.0]) for b in bk[:-1]])
r_btc = np.asarray([ret_index.get(('BTCUSDT', b), 0.0) for b in bk[:-1]])
rows, expo30 = [], None
for beta, model in models.items():
    vals, turns, expo = run_policy(model, bk)
    rows.append({'전략': f'β={beta}', **perf_row(vals, turns), '평균 노출': expo.mean()})
    if beta == 30:
        expo30 = expo
n = min(len(expo30), len(r_ew))
for name, r in [('(a) BTC × β=30 노출 경로', expo30[:n] * r_btc[:n]), ('(b) 등가중 × β=30 노출 경로', expo30[:n] * r_ew[:n]),
                ('(c) BTC 보유', r_btc), ('등가중(무비용 근사)', r_ew)]:
    v = np.concatenate([[1.0], np.cumprod(1 + r)])
    rows.append({'전략': name, **compute_perf(v), '평균 노출': expo30.mean() if '노출 경로' in name else 1.0})
ew_vals, ew_turns = equal_weight_backtest(CryptoPortfolioEnv(buckets=bk, **env_kwargs))
rows.append({'전략': '(d) 등가중(비용 포함)', **perf_row(ew_vals, ew_turns), '평균 노출': 1.0})

holdout = pd.DataFrame(rows)
holdout.insert(0, '구간', f'{bucket_to_date(bk[0])}~{bucket_to_date(bk[-1])}')
holdout.to_csv(CRYPTO_DIR / 'crypto_holdout_eval.csv', index=False, encoding='utf-8-sig')
show = holdout[['전략', '총 수익률', 'Sharpe', 'MDD', '평균 노출']].copy()
for k, f in {'총 수익률': '{:+.2%}', 'Sharpe': '{:.3f}', 'MDD': '{:.2%}', '평균 노출': '{:.1%}'}.items():
    show[k] = show[k].map(f.format)
print(show.to_string(index=False))

get = lambda name: holdout.loc[holdout['전략'] == name, '총 수익률'].iloc[0]
b30, a, b = get('β=30'), get('(a) BTC × β=30 노출 경로'), get('(b) 등가중 × β=30 노출 경로')
rank = holdout[holdout['전략'].str.startswith('β=')].sort_values('총 수익률', ascending=False)['전략'].tolist().index('β=30') + 1
print()
print(f'[사전 등록 판정] β=30 {b30:+.2%} vs (a) {a:+.2%} · (b) {b:+.2%} →',
      '같은 노출 대비 종목 선택 우위가 미지 구간에서도 유지됐다' if (b30 > a and b30 > b) else '유지되지 않았다')
print(f'[보조] 4개 β 중 β=30 순위: {rank}위')
print('저장 완료: crypto_holdout_eval.csv')
